### BK21 data inspection and modeling table construction

In [1]:
import re
import numpy as np
import pandas as pd
from wordfreq import zipf_frequency

In [2]:
stim = pd.read_csv("data_output/bk21_stimuli.csv")
spr = pd.read_csv("data/SPRT_LogLin_216.csv")

print("Stimuli shape:", stim.shape)
print("RT shape:", spr.shape)

print("\nStimuli columns:")
print(stim.columns.tolist())

print("\nRT columns:")
print(spr.columns.tolist())

stim.head()

Stimuli shape: (648, 6)
RT shape: (46092, 18)

Stimuli columns:
['condition', 'sentence', 'prefix', 'target_llm', 'critical_word', 'ITEM']

RT columns:
['SUB', 'TASK_order', 'ITEM', 'position', 'critical_word', 'condition', 'cloze', 'log_cloze', 'c_cloze', 'c_log_cloze', 'cloze_sq', 'log_cloze_sq', 'trigram', 'c_trigram', 'log_trigram', 'c_logtrigram', 'SUM_3RT', 'SUM_3RT_trimmed']


,condition,sentence,prefix,target_llm,critical_word,ITEM
0,HC,"In polluted cities, you can barely breathe the...","In polluted cities, you can barely breathe the",air,air,5
1,MC,The factory built expensive machines to filter...,The factory built expensive machines to filter...,air,air,5
2,LC,They suspected there was something wrong with ...,They suspected there was something wrong with the,air,air,5
3,HC,The soldier decided to quit the army after sch...,The soldier decided to quit the,army,army,6
4,MC,George chose not to join the army after school.,George chose not to join the,army,army,6


### Step 1: Inspect extracted stimuli

At minimum, the stimuli table should contain:

- `ITEM`
- `condition`
- `sentence`
- `critical_word`

Each `(ITEM, condition)` pair should be unique.

In [3]:
print("Missing values in stimuli:")
print(stim.isna().sum())

print("\nDuplicate (ITEM, condition) rows in stimuli:")
print(stim.duplicated(subset=["ITEM", "condition"]).sum())

print("\nNumber of unique ITEM-condition pairs:")
print(stim[["ITEM", "condition"]].drop_duplicates().shape[0])

stim[["ITEM", "condition", "sentence", "critical_word"]].head(10)

Missing values in stimuli:
condition        0
sentence         0
prefix           0
target_llm       0
critical_word    0
ITEM             0
dtype: int64

Duplicate (ITEM, condition) rows in stimuli:
0

Number of unique ITEM-condition pairs:
648


,ITEM,condition,sentence,critical_word
0,5,HC,"In polluted cities, you can barely breathe the...",air
1,5,MC,The factory built expensive machines to filter...,air
2,5,LC,They suspected there was something wrong with ...,air
3,6,HC,The soldier decided to quit the army after sch...,army
4,6,MC,George chose not to join the army after school.,army
5,6,LC,The students were learning about the army afte...,army
6,7,HC,"My sister enjoys poetry, painting, and other f...",art
7,7,MC,"Even though she's blind, she appreciates many ...",art
8,7,LC,They hired the consultant because of her knowl...,art
9,8,HC,They stored all of the Halloween decorations u...,attic


Unique RT lookup table

The RT file contains subject-level rows, so we collapse it to one row per `(ITEM, condition)` for target-word and cloze information.

In [4]:
rt_lookup = (
    spr[["ITEM", "condition", "critical_word", "cloze"]]
    .drop_duplicates()
    .rename(columns={"critical_word": "critical_word_rt"})
)

print("RT lookup shape:", rt_lookup.shape)
rt_lookup.head()

RT lookup shape: (648, 4)


,ITEM,condition,critical_word_rt,cloze
0,5,LC,couch,0.000000
1,6,LC,zoo,0.000000
2,7,LC,stump,0.044944
3,8,LC,drought,0.000000
4,9,LC,small,0.011111


In [5]:
check_df = stim.merge(
    rt_lookup,
    on=["ITEM", "condition"],
    how="left"
)

print("Merged shape:", check_df.shape)
print("\nMissing RT matches:")
print(check_df["critical_word_rt"].isna().sum())

check_df["critical_word"] = check_df["critical_word"].astype(str).str.strip().str.lower()
check_df["critical_word_rt"] = check_df["critical_word_rt"].astype(str).str.strip().str.lower()

check_df["target_match"] = check_df["critical_word"] == check_df["critical_word_rt"]

print("\nTarget match counts:")
print(check_df["target_match"].value_counts(dropna=False))

check_df.loc[~check_df["target_match"], ["ITEM", "condition", "sentence", "critical_word", "critical_word_rt"]].head(20)

Merged shape: (648, 8)

Missing RT matches:
0

Target match counts:
target_match
False    648
Name: count, dtype: int64


,ITEM,condition,sentence,critical_word,critical_word_rt
0,5,HC,"In polluted cities, you can barely breathe the...",air,couch
1,5,MC,The factory built expensive machines to filter...,air,couch
2,5,LC,They suspected there was something wrong with ...,air,couch
3,6,HC,The soldier decided to quit the army after sch...,army,zoo
4,6,MC,George chose not to join the army after school.,army,zoo
5,6,LC,The students were learning about the army afte...,army,zoo
6,7,HC,"My sister enjoys poetry, painting, and other f...",art,stump
7,7,MC,"Even though she's blind, she appreciates many ...",art,stump
8,7,LC,They hired the consultant because of her knowl...,art,stump
9,8,HC,They stored all of the Halloween decorations u...,attic,drought


Create lexical controls from the validated critical word

Compute:

- `word_length`
- `log_freq` using Zipf frequency

In [ ]:
df = check_df.copy()

df["critical_word"] = df["critical_word_rt"]

df = df.drop(columns=["critical_word_rt"])

df.head()



In [ ]:
df["word_length"] = df["critical_word"].astype(str).str.len()

df["log_freq"] = df["critical_word"].apply(
    lambda w: zipf_frequency(str(w), "en")
)

df[["critical_word", "word_length", "log_freq"]].head(10)